In [2]:
from sqlalchemy import create_engine, text
import pandas as pd
import os

In [3]:
import sys
sys.path.append("../pipelines")
from aq import *

print("Lot :", BATCH, "| date de référence :", DATE_REF)
print(lire("SELECT VERSION() AS version, @@collation_server AS collation"))

Lot : L-2026-10-01 | date de référence : 2026-10-01
                  version          collation
0  11.2.6-MariaDB-ubu2204  utf8mb4_nopad_bin


In [4]:
# US-01 — Inventaire des sources cif et crm

# 1.1 Tables, colonnes et clés primaires
cols = lire("""
    SELECT table_schema AS base, table_name AS `table`, COUNT(*) AS colonnes
    FROM information_schema.columns
    WHERE table_schema IN ('cif', 'crm')
    GROUP BY table_schema, table_name
""")

cles = lire("""
    SELECT table_schema AS base, table_name AS `table`,
           GROUP_CONCAT(column_name ORDER BY ordinal_position) AS cle_primaire
    FROM information_schema.key_column_usage
    WHERE table_schema IN ('cif', 'crm') AND constraint_name = 'PRIMARY'
    GROUP BY table_schema, table_name
""")

inventaire = cols.merge(cles, how="left", on=["base", "table"])

# 1.2 Comptage exact des lignes
inventaire["lignes"] = [
    int(lire(f"SELECT COUNT(*) AS n FROM {b}.{t}").n[0])
    for b, t in zip(inventaire.base, inventaire.table)
]

print(inventaire.sort_values(["base", "table"]).to_string(index=False))

# 1.3 Tables sans clé primaire
print(
    "\nTables sans clé primaire :",
    inventaire[inventaire.cle_primaire.isna()][["base", "table"]].values.tolist()
)

base        table  colonnes cle_primaire  lignes
 cif      clients        10      column0   40000
 cif       compte         4      column0   45000
 cif         pays         2    code_pays     250
 crm       client        13    client_id  100000
 crm      contrat         9   contrat_id  100000
 crm     sinistre        10  sinistre_id  100000
 crm type_contrat         3    code_type      12

Tables sans clé primaire : []


In [24]:
# US-02
tables_bronze = lire("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'bronze'
    ORDER BY table_name
""")

print(tables_bronze.to_string(index=False))

      table_name
     cif_clients
      cif_compte
        cif_pays
      crm_client
     crm_contrat
    crm_sinistre
crm_type_contrat


In [25]:
%run ../pipelines/01_ingest_bronze.py

In [26]:
for table in tables_bronze["table_name"]:
    resultat = lire(f"""
        SELECT COUNT(*) AS nb_lignes
        FROM bronze.{table}
        WHERE _batch_id = :batch
    """, batch=BATCH)

    print(f"{table} : {resultat['nb_lignes'].iloc[0]} lignes")

cif_clients : 40000 lignes
cif_compte : 45000 lignes
cif_pays : 250 lignes
crm_client : 100000 lignes
crm_contrat : 100000 lignes
crm_sinistre : 100000 lignes
crm_type_contrat : 12 lignes


In [27]:
lire("""
    SELECT COUNT(*) AS nb_lignes,
           SUM(_ingested_at IS NULL) AS dates_manquantes
    FROM bronze.crm_client
    WHERE _batch_id = :batch
""", batch=BATCH)

,nb_lignes,dates_manquantes
0,100000,0


In [28]:
# US-03 : Porte B — Réconciliation source → bronze (RQ-021)

def somme_controle(base, table, ou=""):
    colonnes = lire("""
        SELECT column_name
        FROM information_schema.columns
        WHERE table_schema = :s
          AND table_name = :t
          AND column_name NOT LIKE '\\_%'
          AND column_name <> 'bronze_row_id'
        ORDER BY ordinal_position
    """, s=base, t=table).column_name.tolist()

    liste = ", ".join(colonnes)

    r = lire(f"""
        SELECT COUNT(*) AS n,
               COALESCE(SUM(CRC32(CONCAT_WS('|', {liste}))), 0) AS crc
        FROM {base}.{table} {ou}
    """)

    return int(r.n[0]), int(r.crc[0])

In [29]:
somme_controle("crm", "client")
somme_controle("bronze", "crm_client", f"WHERE _batch_id = '{BATCH}'")

(100000, 214577921396822)

In [30]:
print("Source :", somme_controle("crm", "client"))
print("Bronze :", somme_controle("bronze", "crm_client", f"WHERE _batch_id = '{BATCH}'"))

Source : (100000, 214577921396822)
Bronze : (100000, 214577921396822)


In [31]:
SOURCES = [("crm", "client", "crm_client"), ("crm", "contrat", "crm_contrat"), ("crm", "type_contrat", "crm_type_contrat"), ("crm", "sinistre", "crm_sinistre"),
           ("cif", "clients", "cif_clients"), ("cif", "compte", "cif_compte"), ("cif", "pays", "cif_pays")]

In [32]:
porte_b = []
for base, table, cible in SOURCES:
    n_src, crc_src = somme_controle(base, table)
    n_brz, crc_brz = somme_controle("bronze", cible, f"WHERE _batch_id = '{BATCH}'")
    ecarts = abs(n_src - n_brz) + int(crc_src != crc_brz)
    porte_b.append(mesurer("BRONZE", "RQ-021", cible, n_src, ecarts))
    # RQ-022 : chaque ligne porte lot, source, horodatage et empreinte
    meta = lire(f"""SELECT COUNT(*) AS n, COALESCE(SUM(_batch_id IS NULL OR _batch_id = '' OR _source_system IS NULL OR _source_system = ''
               OR _ingested_at IS NULL OR _row_hash IS NULL OR _row_hash = ''), 0) AS ko FROM bronze.{cible} WHERE _batch_id = '{BATCH}'""")
    porte_b.append(mesurer("BRONZE", "RQ-022", cible, int(meta.n[0]), int(meta.ko[0])))
print(pd.DataFrame(porte_b).sort_values(["rule_id", "table"]).to_string(index=False))   # aucun écart attendu

rule_id couche            table  testees  echecs  conformite_pct
 RQ-021 BRONZE      cif_clients    40000       0           100.0
 RQ-021 BRONZE       cif_compte    45000       0           100.0
 RQ-021 BRONZE         cif_pays      250       0           100.0
 RQ-021 BRONZE       crm_client   100000       0           100.0
 RQ-021 BRONZE      crm_contrat   100000       0           100.0
 RQ-021 BRONZE     crm_sinistre   100000       0           100.0
 RQ-021 BRONZE crm_type_contrat       12       0           100.0
 RQ-022 BRONZE      cif_clients    40000       0           100.0
 RQ-022 BRONZE       cif_compte    45000       0           100.0
 RQ-022 BRONZE         cif_pays      250       0           100.0
 RQ-022 BRONZE       crm_client   100000       0           100.0
 RQ-022 BRONZE      crm_contrat   100000       0           100.0
 RQ-022 BRONZE     crm_sinistre   100000       0           100.0
 RQ-022 BRONZE crm_type_contrat       12       0           100.0


In [33]:
# Épreuve de l'écart volontaire : sur une COPIE, jamais sur bronze elle-même, et sans écrire le résultat dans dq.dq_history
executer("DROP TABLE IF EXISTS bronze.crm_client_test")
executer("CREATE TABLE bronze.crm_client_test LIKE bronze.crm_client")
executer("INSERT INTO bronze.crm_client_test SELECT * FROM bronze.crm_client")
# À vous : altérez la copie (un UPDATE ... LIMIT ... ou une suppression, un doublon inséré), par exemple :
executer("DELETE FROM bronze.crm_client_test LIMIT 1")
n_t, crc_t = somme_controle("bronze", "crm_client_test")
n_s, crc_s = somme_controle("crm", "client")
print("copie contre source : écarts =", abs(n_s - n_t) + int(crc_s != crc_t), "(0 tant que la copie n'est pas altérée)")
executer("DROP TABLE bronze.crm_client_test")

copie contre source : écarts = 2 (0 tant que la copie n'est pas altérée)
